## Импорты


In [ ]:
# запускать ноутбук в корне проекта, чтобы работали относительные пути
from pathlib import Path
from datetime import datetime
import os
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src/qml_hybrid").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))

from torch import nn
from IPython.display import display
import matplotlib.pyplot as plt
from qml_hybrid.classical import classical_architecture
from qml_hybrid.model import ModelConfig
from qml_hybrid.noisy import NoiseConfig, NoisyHybridClassifier
from qml_hybrid.vision_data import load_vision_data
from qml_hybrid.vision_training import CPUConfig, fit_vision, report_vision_study, load_noisy_checkpoint
from qml_hybrid.vision_plots import plot_vision_comparison, comparison_best_metrics
from qml_hybrid.vision_notebook import draw_vision_circuit, plot_run
from qml_hybrid.training import seed_everything


## Настройка обучения


In [ ]:
# MNIST / Fashion MNIST / CIFAR-10; limit=0 — все объекты.
data_settings = dict(
    name="CIFAR-10", image_size=32, grayscale=False, seed=42,
    val_size=0.0,  # CIFAR: 0; остальные: 0 или доля train, например 0.15.
    train_limit=1000, val_limit=250, test_limit=100,
)
data = load_vision_data(root=ROOT / "data/vision", **data_settings)
display(data.distribution())


In [ ]:
# Энкодер → 8 признаков → квантовый блок → 8 признаков → голова.
encoder = nn.Sequential(
    nn.Linear(data.train_x.shape[1], 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 32), nn.Tanh(),
    nn.Linear(32, 8), nn.Tanh(),
)

head = nn.Sequential(
    nn.Linear(8, 16), nn.ReLU(), 
    nn.Linear(16, len(data.classes))
)

classical = classical_architecture(encoder, head)
model_config = ModelConfig(
    n_features=data.train_x.shape[1], n_classes=len(data.classes), **classical,
    quantum_depth=4, embedding="RX", rotation_gates=("RY", "RX"),
    entanglement="linear",  # linear / ring / brickwork / brickwork3 / full / none / custom
    custom_edges=((0, 1), (1, 2), (2, 7)),  # Нумерация кубитов: 0–7.
    entangler="CZ", measurement="Z", initial_hadamard=False, reupload=True,
    trainable_embedding=True, embedding_parameterization="per_rotation",
    shots=1000, measurement_seed=42,  # None — аналитические средние.
    measurement_mode="parameter_shift",  # sampled_backprop / analytic_train / reinforce
)
noise_config = NoiseConfig(
    backend="torch_density",  # torch_density / pennylane_mixed / pennylane_statevector
    precision="float32",  # float32 / float64
    enabled=True,  # Для pennylane_statevector: False.
    t1_us=14.0, t2_us=4.33,  # Одно число или восемь значений; T2 ≤ 2*T1.
    single_gate_us=0.05, two_gate_us=0.05, readout_idle_us=0.0,
    gate_errors=False, single_gate_fidelity=0.99, two_gate_fidelity=0.90,
    gate_error_seed=42,
)
assert noise_config.backend in ("torch_density", "pennylane_mixed", "pennylane_statevector")


In [ ]:
seeds = [11,22,33]
cpu = CPUConfig(workers=2, threads_per_worker=1, microbatch_size=2, parallel_mode="split_batch")
training = dict(epochs=50, batch_size=500, patience=100, min_delta=0.0,
                lr=6e-3, quantum_lr=6e-3,
                regularization=dict(classical_l2=0.0, quantum_periodic=0.0, embedding_l2=0.0))
mode = dict(quantum=True, train_classical=True, train_quantum=True)
output_name = "noisy_vision"
checkpoint_path = ""  # Путь к checkpoint для восстановления; иначе новая модель.
continue_current_model = False  # True — продолжить текущие веса, только один seed.

seed_everything(seeds[0])
if checkpoint_path:
    model, checkpoint = load_noisy_checkpoint(checkpoint_path)
    model_config, noise_config = model.config, model.noise
    assert noise_config.backend in ("torch_density", "pennylane_mixed", "pennylane_statevector")
else:
    model = NoisyHybridClassifier(model_config, noise_config)
model.set_mode(**mode)
print(model)
print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Квантовая схема


In [ ]:
figure, _ = draw_vision_circuit(model, ROOT / "runs/quantum_circuit.png")
display(figure)
plt.close(figure)


## Запуск


In [ ]:
# Одна эпоха — весь train; без val тест считается после каждой эпохи.
if not seeds or (continue_current_model and len(seeds) != 1):
    raise ValueError("Нужен список seed; для продолжения — один seed")
if Path(output_name).name != output_name or output_name in ("", ".", ".."):
    raise ValueError("output_name: простое имя каталога")
study_folder = ROOT / "runs" / output_name / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results = []
for seed in seeds:
    if not continue_current_model:
        seed_everything(seed)
        model = NoisyHybridClassifier(model_config, noise_config)
    model.set_mode(**mode)
    result = fit_vision(model, data, study_folder / f"seed_{seed}", cpu=cpu, seed=seed,
                        initialization="continued" if continue_current_model else "independent", **training)
    results.append(result)
    plot_run(result)
print(study_folder)


## Отрисовка результатов


In [ ]:
# Графики сохранённых запусков; обучение не требуется.
study_name = "20261021_233017_092277_sv_50k_10k_10k_copy"
study_folder = ROOT / "runs/noisy_vision" / study_name

collect_paths = sorted(
    p.parent for p in study_folder.glob("seed_*/metrics.json")
)

band = "sd"  # стандартное отклонение между seed

print("Найдено запусков:", len(collect_paths))
assert collect_paths, "Папки seed_* с metrics.json не найдены"

legend_loc = "lower right"  # upper right / upper left / lower right / lower left / best; None — скрыть
legend_on = "right"  # left — слева; right — справа; both — оба; none — скрыть
legend_ncol = 1  # число колонок легенды

# panel_positions = ((-0.12, 1.04), (-0.12, 1.04))  # (x, y) для (а) и (б); None — скрыть

summary = report_vision_study(
    collect_paths,
    study_folder / "report_selected",
    band=band,
    seeds=None,  # [11,22,33,44,55],  # укажите нужные seed
    legend_loc=legend_loc, legend_on=legend_on,
    legend_ncol=legend_ncol,
    allow_mixed_protocols=True,
    # panel_positions=panel_positions,
)
display(summary)


In [ ]:
comparison_studies = {


    "3 000": ROOT / "runs/noisy_vision/20261022_135814_308230_sv_3k_10k_1k",
    "10 000": ROOT / "runs/noisy_vision/20261022_115424_187192_sv_10k_10k_2.5k",
    "50 000": ROOT / "runs/noisy_vision/20261021_233017_092277_sv_50k_10k_10k_copy",

}
comparison_seeds = None     # все seed; либо [11, 22, 33, 44]
comparison_band = "sd"     # "sd" или "ci95"
comparison_split = "auto"  # val при наличии, иначе test; можно явно "val" или "test"
comparison_epochs = None   # все эпохи; либо, например, 40

comparison_xlim = (0,50)     # диапазон оси эпох: например, (0, 40) или (10, 100)

comparison_ylim_loss = (1.1, 2.4)      # диапазон потерь: например, (0, 2.5)
comparison_ylim_accuracy = (0.09, 0.55)  # диапазон точности: например, (0, 1)

comparison_test_reference = None
comparison_test_epoch = 100  # точная эпоха ориентира; None — последняя доступная внутри xlim
comparison_reference_seeds = None  # все seed или [11, 22, 33]
comparison_reference_label = "Тестовая выборка, предел (50.000 изображений)"

comparison_legend_loc = "lower left" 
comparison_legend_on = "left"
comparison_legend_ncol = 1 

comparison_allow_mixed_protocols = {"50 000": True}

comparison_panel_positions = ((-0.12, 1.04), (-0.12, 1.04))  # (x, y) для (а) и (б); None — скрыть

comparison = plot_vision_comparison(
    comparison_studies,
    ROOT / "runs/noisy_vision/plots_comparison",
    seeds=comparison_seeds, band=comparison_band,
    eval_split=comparison_split, max_epochs=comparison_epochs, xlim=comparison_xlim,
    ylim_loss=comparison_ylim_loss, ylim_accuracy=comparison_ylim_accuracy,
    test_reference=comparison_test_reference, test_epoch=comparison_test_epoch,
    reference_seeds=comparison_reference_seeds, reference_label=comparison_reference_label,
    legend_loc=comparison_legend_loc, legend_on=comparison_legend_on,
    legend_ncol=comparison_legend_ncol,
    allow_mixed_protocols=comparison_allow_mixed_protocols,
    panel_positions=comparison_panel_positions,
)
display(comparison["curves"])
print("Графики PNG/SVG и данные CSV:", comparison["output"])


In [ ]:
best_metrics = comparison_best_metrics(comparison)
display(best_metrics.style.format({
    "Величина разброса (±)": "{:.4f}",
    "Значение": "{:.4f}", "Нижняя граница": "{:.4f}", "Верхняя граница": "{:.4f}",
}, na_rep="—"))
best_metrics.to_csv(Path(comparison["output"]) / "best_metrics.csv", index=False, encoding="utf-8-sig")


## QPU time est

In [ ]:
n_shots = 1000

t_PQC = 1200
t_readout = 1000
t_equilibrium = 5000

t_waveform = 10.0 * 1e6

n_1q = 96

t_available = 100 # hours
n_epochs = 40 # проход по всем данным

train_size = 50000


In [ ]:
t_1elem = (n_shots * (t_PQC + t_readout + t_equilibrium) + t_waveform) * (2*n_1q + 1)
print(f'1 element time: {t_1elem * 1e-9:.2f} s')
print(f'train time for epoch: {t_1elem * 1e-9 * train_size / 3600 :.2f} h \n')

print(f'Available time: {t_available:.2f} h')
print(f'In available time, you can train for {t_available / (t_1elem * 1e-9 * train_size / 3600):.2f} epochs. \n')

print(f'elements to match {n_epochs} epochs in available time: {t_available / (n_epochs * (t_1elem * 1e-9 / 3600)):.2f} elements')
